Example 1

In [1]:
import rapidsegment as rs
from prettytable import PrettyTable
import pandas as pd
from rapidsegment import UniversalDataLoader
from rapidsegment import StrategicSegmentBuilder
from rapidsegment import StrategicSegmentScore
import duckdb

In [2]:
print(f"RapidSegment version: {rs.__version__}")

RapidSegment version: 1.2.1


LOAD DATA

In [3]:
data = UniversalDataLoader(file_path=r"/workspaces/RapidSegment/Examples/bank-full.csv", ).load()
print(f"Loaded as {type(data)} table for better performance ")
data.to_pandas().head()

2026-08-14 03:48:22,011 | INFO     | [data_loader.py:147] | 📂 Loading file: /workspaces/RapidSegment/Examples/bank-full.csv (extension: .csv)


Loaded as <class 'pyarrow.lib.Table'> table for better performance 


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,Target
0,58.0,management,married,tertiary,no,2143.0,yes,no,unknown,5.0,may,261.0,1.0,-1.0,0.0,unknown,no
1,44.0,technician,single,secondary,no,29.0,yes,no,unknown,5.0,may,151.0,1.0,-1.0,0.0,unknown,no
2,33.0,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5.0,may,76.0,1.0,-1.0,0.0,unknown,no
3,47.0,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5.0,may,92.0,1.0,-1.0,0.0,unknown,no
4,33.0,unknown,single,unknown,no,1.0,no,no,unknown,5.0,may,198.0,1.0,-1.0,0.0,unknown,no


Encoding Target Variable into Binary Format


In [4]:
import duckdb
conn = duckdb.connect(database=':memory:', read_only=False)
conn.register("df", data)
conn.execute("CREATE OR REPLACE TABLE mod_df AS (SELECT *EXCLUDE(Target), CASE WHEN Target = 'yes' THEN 1 ELSE 0 END AS Target FROM df)")
conn.execute("SELECT * FROM mod_df").fetchdf().head()
mod_data = conn.execute("SELECT * FROM mod_df").to_arrow_table()

In [5]:
help(StrategicSegmentBuilder)

Help on class StrategicSegmentBuilder in module rapidsegment.builder:

class StrategicSegmentBuilder(builtins.object)
 |  StrategicSegmentBuilder(target: str, n_jobs: int = -1, min_sample_size: int = 1000, min_lift: float = 1.5, min_events: int = 100, top_n_vars: int = 15, max_segments: int = 10, max_feature_reuse: int = 1, param_grid: Optional[Dict[str, List[Any]]] = None, enable_diversity: bool = False, enable_1way: bool = True, enable_2way: bool = True, enable_3way: bool = True, feature_groups: Optional[Dict[str, List[str]]] = None, ignore_features: Optional[List[str]] = None, sort_priority: str = 'rate_lift_count', binning_method: str = 'optimal', naive_bins: int = 5, max_expansion_hops: int = 0, selection_metric: str = 'iv', expand_log_mode: str = 'none', db_path: Optional[str] = None, db_temp_dir: Optional[str] = None) -> None
 |
 |  Extracts hierarchical, predictive segments from tabular data.
 |
 |  The extraction is sequential:
 |      - At each step, the best rule (by lift an

1. Rule Mining using OptimalBinning as binning strategy (binning_method="optimal")
2. IV as variable selection metric (selection_metric='iv')
3. Repeated use of features are allowed (max_feature_reuse = 5)
4. Segment ranks on #event -> %response rate -> lift

Setup Experiment

In [6]:
param_grid = {'min_sample_size': [20000, 15000, 10000, 5000, 500],'min_lift': [3.0, 2.0, 1.5]}
builder = StrategicSegmentBuilder(target = 'Target',
                                  min_sample_size = 100,
                                  min_lift = 1.0, 
                                  min_events  = 50, 
                                  top_n_vars = 10,
                                  max_segments = 10,
                                  param_grid = param_grid,
                                  enable_diversity = False, 
                                  max_feature_reuse = 5, 
                                  enable_1way = True,
                                  enable_2way = True,
                                  enable_3way = True,
                                  feature_groups = None,
                                  ignore_features = None,
                                  sort_priority = 'events_rate_lift',
                                  binning_method="optimal",
                                  selection_metric='iv')

Start Experiment

In [7]:
segments_df = builder.extract_segments(mod_data)

2026-08-14 03:48:22,263 | INFO     | [builder.py:1080] | 🚀 Starting hierarchical segment extraction...
2026-08-14 03:48:22,265 | INFO     | [builder.py:1099] | 📂 Created temporary disk-backed DB at: experiments/segmentation_20260814_3d20a4aa.duckdb
2026-08-14 03:48:22,288 | INFO     | [builder.py:1114] | ⚙️ DuckDB Configured for Disk Spilling: Threads=4/4, MemoryLimit=12GB, TempDir=None
2026-08-14 03:48:22,289 | INFO     | [builder.py:1118] | 📊 Sort priority: events_rate_lift
2026-08-14 03:48:22,289 | INFO     | [builder.py:1119] | 📦 Binning method: optimal (naive_bins=5)
2026-08-14 03:48:22,500 | INFO     | [builder.py:1174] | 📊 Dynamic Grid Search Enabled: 15 configurations.
2026-08-14 03:48:22,501 | INFO     | [builder.py:1182] | 🔒 Locking Original Base Rate: 11.70%
2026-08-14 03:48:22,502 | INFO     | [builder.py:1208] | 🔄 Iteration 1 | Remaining Volume: 45,211 | Base Rate: 11.70%
2026-08-14 03:48:22,503 | INFO     | [builder.py:279] | 🔍 Computing IV and bins for 16 features...
202

Build Experiment Results

In [8]:
final_eval = builder.evaluate_final_coverage(mod_data)

2026-08-14 03:48:25,887 | INFO     | [builder.py:1617] | 📊 Evaluating final hierarchical coverage on original data...


Final Segment Report

In [9]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |        lift        | cumulative_sample_capture | cumulative_event_capture |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
|   1.0   |    5818.0   |     760.0     | 13.062908215881746 | 11.698480458295547 | 12.868549689234921 | 1.1166329047990728 |     12.868549689234921    |    14.369446020041595    |
|   0.0   |   39393.0   |     4529.0    | 11.496966466123423 | 11.698480458295547 | 87.13145031076508  | 0.9827743446774553 |           100.0           |          100.0           |
+---------+-------------+---------------+--------------------+--------------------+------------

Segment SQL definition

In [10]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   balance=[2324.50, inf) & poutcome=[unknown]
SQL Filter: (balance >= 2324.50) AND (poutcome IN ('unknown'))
--------------------------------------------------


Segment Meta informations

In [11]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(segments_df).columns)
for _, row in pd.DataFrame(segments_df).iterrows():
    table.add_row(list(row))
print(table)

+------------+---------------------------------------------+----------------------------------------------------+-------+--------------------+--------------------+--------------------------+-----------------------+
| segment_id |                 rule_string                 |                     sql_filter                     | count |        rate        |        lift        | meta_applied_sample_size | meta_applied_min_lift |
+------------+---------------------------------------------+----------------------------------------------------+-------+--------------------+--------------------+--------------------------+-----------------------+
|     1      | balance=[2324.50, inf) & poutcome=[unknown] | (balance >= 2324.50) AND (poutcome IN ('unknown')) |  5818 | 13.062908215881746 | 1.1166329047990728 |           5000           |          3.0          |
+------------+---------------------------------------------+----------------------------------------------------+-------+-------------------

Selection Audit Trail for Variable

In [12]:
builder.explain_feature_journey("campaign")

📌 AUDIT TRAIL FOR FEATURE: 'campaign'

[Iteration 1]
  • Current dynamic IV   : 0.0850
  • Previous times used  : 0
  • Selection Status     : Excluded (Outside Top N Features by Score)
  • Winner this round    : balance=[2324.50, inf) & poutcome=[unknown] (Variables: ['balance', 'poutcome'])

[Iteration 2]
  • Current dynamic IV   : 0.0909
  • Previous times used  : 0
  • Selection Status     : Excluded (Outside Top N Features by Score)


Preparing the dataset for scoring and decile banding.
- Only score when atleast 10 segments are found

In [13]:
conn = duckdb.connect()
conn.register("predicted", mod_data)
predicted = conn.query("""
                        SELECT *, 
                        CASE WHEN (contact IN ('cellular')) AND (housing IN ('no'))
                        THEN 1 ELSE 0 END AS seg_1,
                        CASE WHEN (duration >= 551.50)
                        THEN 1 ELSE 0 END AS seg_2,
                        CASE WHEN (month IN ('feb', 'dec', 'sep', 'oct', 'mar'))
                        THEN 1 ELSE 0 END AS seg_3,
                        CASE WHEN (pdays >= 0.00 AND pdays < 213.00) AND (poutcome IN ('other', 'success'))
                        THEN 1 ELSE 0 END AS seg_4,                                                                                       
                        ROW_NUMBER() OVER () AS ID,
                        FROM predicted
""").df()
conn.close()

In [14]:
predicted.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,...,campaign,pdays,previous,poutcome,Target,seg_1,seg_2,seg_3,seg_4,ID
0,58.0,management,married,tertiary,no,2143.0,yes,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,1
1,44.0,technician,single,secondary,no,29.0,yes,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,2
2,33.0,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,3
3,47.0,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,4
4,33.0,unknown,single,unknown,no,1.0,no,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,5


Score the segments on the dataset and create decile bands


In [15]:
scorer = StrategicSegmentScore(
    target_col="Target",
    primary_key="ID",
    segment_cols=["seg_1","seg_2",'seg_3','seg_4'],
)

Export Segment Score as JSON

In [16]:
model_artifact = scorer.calculate_and_export_weights(predicted)

2026-08-14 03:48:26,177 | INFO     | [scorer.py:71] | 🚀 Initialising out‑of‑core DuckDB scorecard engine...
2026-08-14 03:48:26,476 | INFO     | [scorer.py:113] | 📊 Computing scorecard weights...
2026-08-14 03:48:26,477 | WARNING  | [scorer.py:157] | ⚠️ DECILE RESOLUTION WARNING: Only 4 distinct non-zero score values found across 4 segments. Splitting into 10 deciles will produce repeated thresholds (e.g., top 5 deciles may have identical scores). For smooth decile ranking, ensure the builder discovers at least 10 distinct segments (increase `max_segments`). Consider interpreting results as tiers rather than deciles.
2026-08-14 03:48:26,478 | INFO     | [scorer.py:171] | ⚡ Scoring population natively via SQL engine...
2026-08-14 03:48:26,501 | INFO     | [scorer.py:191] | 📉 Dataset Zero‑Inflation Rate: 88.30%
2026-08-14 03:48:26,501 | INFO     | [scorer.py:196] | 📈 Calibrating deciles across active populations...
2026-08-14 03:48:26,506 | INFO     | [scorer.py:244] | ✅ Scorecard export

View segment score and create final Decile based summary

In [17]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 20
Segment: seg_2 | Weight: 46
Segment: seg_3 | Weight: 30
Segment: seg_4 | Weight: 48


In [18]:
model_artifact.get("decile_min_thresholds")

{'1': 144,
 '2': 66,
 '3': 50,
 '4': 46,
 '5': 46,
 '6': 20,
 '7': 20,
 '8': 20,
 '9': 20,
 '10': 20}

In [19]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 20 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 46 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 30 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 48 ELSE 0 END AS seg_4_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted ) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=144 THEN 1
                    WHEN total_weight >= 66 THEN 2
                    WHEN total_weight >= 50 THEN 3
                    WHEN total_weight >= 46 THEN 4
                    WHEN total_weight >= 46 THEN 5
                    WHEN total_weight >= 20 THEN 6
                    WHEN total_weight >= 20 THEN 7
                    WHEN total_weight >= 20 THEN 8
                    WHEN total_weight >= 20 THEN 9
                    WHEN total_weight >= 20 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [20]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(Target) AS events, 
                    (SUM(Target)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,0,25008,618.0,2.471209
1,1,47,37.0,78.723404
2,2,3003,1600.0,53.280053
3,3,1863,507.0,27.214171
4,4,3340,1279.0,38.293413
5,6,11950,1248.0,10.443515
